In [1]:
import os
import json
import shutil
import random
from pathlib import Path
from dataclasses import dataclass, asdict
from coordinator import Coordinator
from merge_worker import MergeWorker, WorkerStats, Message

def run_test(data_a, data_b, test_name):
    print(f"Running Test: {test_name}")
    
    # Setup paths
    paths = ["A_to_B.msg", "B_to_A.msg", "output.txt", "state_a.json", "state_b.json"]
    for p in paths:
        if os.path.exists(p): os.remove(p)
    
    Path("output.txt").touch()

    # Initialize Workers
    worker_a = MergeWorker("A", data_a, Path("B_to_A.msg"), Path("A_to_B.msg"), 
                           Path("output.txt"), Path("state_a.json"))
    worker_b = MergeWorker("B", data_b, Path("A_to_B.msg"), Path("B_to_A.msg"), 
                           Path("output.txt"), Path("state_b.json"))

    # Run Coordinator
    coord = Coordinator(worker_a, worker_b)
    result = coord.run()

    # Verify Correctness
    with open("output.txt") as f:
        output_data = [int(line.strip()) for line in f]
    
    expected = sorted(data_a + data_b)
    is_correct = output_data == expected
    
    # Calculate Balance
    comp_a = result["stats_a"].comparisons
    comp_b = result["stats_b"].comparisons
    balance = ( min(comp_a, comp_b)) / max(comp_a, comp_b) 

    print(f"Success: {is_correct}")
    print(f"Steps: {result['total_steps']}")
    print(f"Messages: {result['stats_a'].messages_sent + result['stats_b'].messages_sent}")
    print(f"Balance: {balance:.2f} (A:{comp_a}, B:{comp_b})")

    print(f"Size A: {len(data_a)}, Size B: {len(data_b)}")
    print(result)

    print("-" * 30 + "\n")


In [2]:
# Test 1: Even-odd Ordered
run_test(list(range(0, 100, 2)), list(range(1, 100, 2)), "Even-Odd")

# Test 2: Large Disjoint Ordered
run_test(list(range(1000, 6000)), list(range(0, 999)), "Disjoint")

# Test 3: Small set 
run_test([1, 5, 9, 13, 17, 21], [2, 6, 10, 14, 18, 22], "Small")

# Test 4: Repeats Small set
run_test([1, 5, 9, 13, 17, 21], [13, 17, 21, 2, 6, 10, 14, 18, 22], "Small Repeated")

# Test 4: Random set
run_test([random.randint(-100, 100) for _ in range(random.randint(1, 100))], 
         [random.randint(-100, 100) for _ in range(random.randint(1, 100))], "Random")

Running Test: Even-Odd
Success: True
Steps: 14
Messages: 13
Balance: 1.00 (A:50, B:50)
Size A: 50, Size B: 50
{'success': True, 'total_steps': 14, 'stats_a': WorkerStats(comparisons=50, messages_sent=7, messages_received=6, values_output=50), 'stats_b': WorkerStats(comparisons=50, messages_sent=6, messages_received=7, values_output=50)}
------------------------------

Running Test: Disjoint
Success: True
Steps: 608
Messages: 607
Balance: 1.00 (A:2999, B:3000)
Size A: 5000, Size B: 999
{'success': True, 'total_steps': 608, 'stats_a': WorkerStats(comparisons=2999, messages_sent=304, messages_received=303, values_output=2999), 'stats_b': WorkerStats(comparisons=3000, messages_sent=303, messages_received=304, values_output=3000)}
------------------------------

Running Test: Small
Success: True
Steps: 10
Messages: 9
Balance: 1.00 (A:6, B:6)
Size A: 6, Size B: 6
{'success': True, 'total_steps': 10, 'stats_a': WorkerStats(comparisons=6, messages_sent=5, messages_received=4, values_output=6),